# Ejercicio 3: consultas directas sobre archivos Parquet

En este notebook consultamos los Parquet de 2026 sin importarlos a ninguna tabla, usando `read_parquet` con globs sobre `data/raw`. Cada consulta vive en su propio archivo dentro de `sql/ejercicio_03/` y aquí solo la leemos, la mostramos y la ejecutamos, para que exista en un solo lugar. El análisis completo de cada resultado y las decisiones que tomamos están en `docs/ejercicio_03.md`.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)

SQL = Path("/workspace/sql/ejercicio_03")
con = duckdb.connect()  # en memoria, no se guarda ninguna tabla
con.execute("SET threads = 4")
con.execute("SET memory_limit = '2GB'")
# Este ejercicio documenta los datos de 2026, que eran los unicos cuando lo hicimos.
# Sin esta linea las vistas leen todos los anios que haya en data/raw.
con.execute("SET VARIABLE anios = [2026]")

def correr(nombre):
    """Muestra el SQL del archivo y devuelve su resultado."""
    texto = (SQL / f"{nombre}.sql").read_text(encoding="utf-8")
    print(texto)
    return con.sql(texto).df()

Casi todas las consultas de calidad trabajan sobre una vista llamada `viajes` que une amarillos y verdes con nombres de columna comunes. La vista no copia datos, solo guarda la consulta sobre los Parquet, así que cada vez que se usa vuelve a leer los archivos. Lee todos los años que haya en `data/raw` y respeta la variable de sesión `anios`, que aquí fijamos en 2026 para que los resultados correspondan a los datos con los que hicimos este ejercicio.

In [2]:
con.execute((SQL / "00_vista_viajes.sql").read_text(encoding="utf-8"))
print((SQL / "00_vista_viajes.sql").read_text(encoding="utf-8"))

-- Vista de apoyo: une amarillos y verdes con nombres de columna comunes.
-- No copia datos: es solo una consulta guardada sobre los Parquet, que se
-- vuelve a leer cada vez que se usa la vista. Los viajes se distinguen por tipo.
--
-- Lee todos los anios que haya en data/raw (yellow/*/ y green/*/), de modo que
-- un anio nuevo entra sin tocar esta vista. Para analizar solo algunos anios se
-- define antes la variable de sesion anios; si no se define, se usan todos:
--     SET VARIABLE anios = [2026];
--     SET VARIABLE anios = [2024, 2026];
--
-- Fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet
CREATE OR REPLACE VIEW viajes AS
SELECT *
FROM (
    SELECT
        'yellow' AS tipo,
        filename AS archivo,
        regexp_extract(filename, '(\d{4}-\d{2})', 1) AS mes_archivo,
        CAST(regexp_extract(filename, 'tripdata_(\d{4})', 1) AS INTEGER) AS anio_archivo,
        tpep_pickup_datetime AS pickup,
        tpep_dropoff_datetime AS dropoff,
        * EXCLUDE (file

## 3.1 Cantidad de archivos

Usamos `glob()` para listar los archivos sin abrirlos, que es la forma más barata de saber qué hay disponible.

In [3]:
correr('3_1_cantidad_archivos')

-- 3.1 Cantidad de archivos disponibles.
-- Objetivo: contar los Parquet por tipo de taxi y ver el rango de meses.
-- Fuente: data/raw/*/2026/*.parquet (glob() solo lista archivos, no los lee).
SELECT
    coalesce(regexp_extract(file, '(yellow|green)_tripdata', 1), 'total') AS tipo,
    count(*) AS archivos,
    min(regexp_extract(file, '(\d{4}-\d{2})', 1)) AS primer_mes,
    max(regexp_extract(file, '(\d{4}-\d{2})', 1)) AS ultimo_mes
FROM glob('/workspace/data/raw/*/2026/*.parquet')
GROUP BY ROLLUP (regexp_extract(file, '(yellow|green)_tripdata', 1))
ORDER BY tipo;



,tipo,archivos,primer_mes,ultimo_mes
0,green,8,2026-01,2026-08
1,total,16,2026-01,2026-08
2,yellow,8,2026-01,2026-08


## 3.2 Cantidad de registros

Contamos de dos formas, una leyendo los datos y otra leyendo solo el pie de página de cada Parquet, para comprobar que ambas coinciden.

In [4]:
correr('3_2a_registros_por_tipo_y_mes')

-- 3.2 Cantidad de registros disponibles (conteo real sobre los datos).
-- Objetivo: contar viajes por tipo de taxi y mes de archivo.
-- Fuente: yellow/2026/*.parquet y green/2026/*.parquet con filename = true.
SELECT
    regexp_extract(filename, '(yellow|green)_tripdata', 1) AS tipo,
    regexp_extract(filename, '(\d{4}-\d{2})', 1) AS mes_archivo,
    count(*) AS viajes
FROM read_parquet('/workspace/data/raw/*/2026/*.parquet',
                  union_by_name = true, filename = true)
GROUP BY ALL
ORDER BY tipo, mes_archivo;



,tipo,mes_archivo,viajes
0,green,2026-01,40272
1,green,2026-02,37373
2,green,2026-03,44208
3,green,2026-04,44238
4,green,2026-05,44921
5,green,2026-06,44163
6,green,2026-07,41252
7,green,2026-08,40687
8,yellow,2026-01,3724889
9,yellow,2026-02,3399866


In [5]:
correr('3_2b_registros_segun_metadata')

-- 3.2 Cantidad de registros segun la metadata del pie de pagina de Parquet.
-- Objetivo: obtener el mismo total sin leer los datos y compararlo con 3.2a.
-- Fuente: data/raw/*/2026/*.parquet (solo se lee el footer de cada archivo).
SELECT
    coalesce(regexp_extract(file_name, '(yellow|green)_tripdata', 1), 'total') AS tipo,
    count(*) AS archivos,
    sum(num_rows) AS viajes,
    sum(num_row_groups) AS grupos_de_filas
FROM parquet_file_metadata('/workspace/data/raw/*/2026/*.parquet')
GROUP BY ROLLUP (regexp_extract(file_name, '(yellow|green)_tripdata', 1))
ORDER BY tipo;



,tipo,archivos,viajes,grupos_de_filas
0,green,8,337114.0,8.0
1,total,16,30040469.0,40.0
2,yellow,8,29703355.0,32.0


## 3.3 y 3.4 Columnas y tipos de datos

Comparamos los esquemas de amarillos y verdes lado a lado. Como las fechas se llaman distinto en cada tipo, normalizamos el nombre para poder alinearlas.

In [6]:
correr('3_3_y_3_4_columnas_y_tipos')

-- 3.3 y 3.4 Columnas presentes y tipos de datos, amarillos contra verdes.
-- Objetivo: comparar los esquemas. Las columnas de fecha se llaman distinto
-- (tpep_ contra lpep_), asi que se normaliza el nombre para alinearlas.
-- union_by_name = true une los archivos por nombre, porque algunas columnas
-- no existen en todos los meses.
-- Fuente: yellow/2026/*.parquet y green/2026/*.parquet.
WITH y AS (
    SELECT regexp_replace(column_name, '^tpep_', 'xpep_') AS columna,
           column_name AS nombre_yellow, column_type AS tipo_yellow
    FROM (DESCRIBE SELECT * FROM read_parquet(
        '/workspace/data/raw/yellow/2026/*.parquet', union_by_name = true))
),
g AS (
    SELECT regexp_replace(column_name, '^lpep_', 'xpep_') AS columna,
           column_name AS nombre_green, column_type AS tipo_green
    FROM (DESCRIBE SELECT * FROM read_parquet(
        '/workspace/data/raw/green/2026/*.parquet', union_by_name = true))
)
SELECT
    coalesce(y.columna, g.columna) AS columna,
    y.nomb

,columna,nombre_yellow,tipo_yellow,nombre_green,tipo_green,presencia
0,DOLocationID,DOLocationID,INTEGER,DOLocationID,INTEGER,en ambos
1,PULocationID,PULocationID,INTEGER,PULocationID,INTEGER,en ambos
2,RatecodeID,RatecodeID,BIGINT,RatecodeID,BIGINT,en ambos
3,VendorID,VendorID,INTEGER,VendorID,INTEGER,en ambos
4,cbd_congestion_fee,cbd_congestion_fee,DOUBLE,cbd_congestion_fee,DOUBLE,en ambos
5,congestion_surcharge,congestion_surcharge,DOUBLE,congestion_surcharge,DOUBLE,en ambos
6,extra,extra,DOUBLE,extra,DOUBLE,en ambos
7,fare_amount,fare_amount,DOUBLE,fare_amount,DOUBLE,en ambos
8,improvement_surcharge,improvement_surcharge,DOUBLE,improvement_surcharge,DOUBLE,en ambos
9,mta_tax,mta_tax,DOUBLE,mta_tax,DOUBLE,en ambos


Después revisamos si el esquema cambia entre meses, que es el motivo por el que usamos `union_by_name`.

In [7]:
correr('3_4b_columnas_por_archivo')

-- 3.4 Consistencia del esquema entre meses.
-- Objetivo: ver en cuantos archivos aparece cada columna y con que tipo fisico
-- de Parquet, para detectar cambios de esquema entre meses. Solo se muestran
-- las columnas que no estan en los 8 archivos de su tipo.
-- Fuente: data/raw/*/2026/*.parquet (solo el esquema, no los datos).
SELECT
    regexp_extract(file_name, '(yellow|green)_tripdata', 1) AS tipo,
    name AS columna,
    type AS tipo_parquet,
    count(DISTINCT file_name) AS archivos_con_la_columna,
    min(regexp_extract(file_name, '(\d{4}-\d{2})', 1)) AS desde_mes
FROM parquet_schema('/workspace/data/raw/*/2026/*.parquet')
WHERE name <> 'schema'
GROUP BY ALL
HAVING count(DISTINCT file_name) < 8
ORDER BY tipo, columna;



,tipo,columna,tipo_parquet,archivos_con_la_columna,desde_mes
0,green,request_source,BYTE_ARRAY,3,2026-06
1,yellow,request_source,BYTE_ARRAY,3,2026-06


## 3.5 Muestra de registros

Para que la muestra sea la misma en cada corrida filtramos por una clase de hash en lugar de usar un muestreo aleatorio.

In [8]:
correr('3_5a_muestra_yellow')

-- 3.5 Muestra de registros de taxis amarillos.
-- Objetivo: ver como lucen los datos reales. La muestra es reproducible: toma
-- las filas cuyo hash cae en una clase fija, en lugar de un muestreo aleatorio
-- que cambiaria en cada corrida.
-- Fuente: data/raw/yellow/2026/*.parquet.
SELECT *
FROM read_parquet('/workspace/data/raw/yellow/2026/*.parquet', union_by_name = true)
WHERE hash(tpep_pickup_datetime, PULocationID, DOLocationID, total_amount) % 4000000 = 0
ORDER BY tpep_pickup_datetime;



,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source
0,2,2026-01-11 18:21:07,2026-01-11 18:28:18,1,1.03,1,N,237,230,1,8.60,0.00,0.5,2.52,0.0,1.0,15.12,2.5,0.0,0.00,None
1,2,2026-02-09 10:58:35,2026-02-09 11:00:22,1,0.57,1,N,75,238,1,5.10,0.00,0.5,1.32,0.0,1.0,7.92,0.0,0.0,0.00,None
2,2,2026-03-16 17:19:54,2026-03-16 17:28:38,1,1.02,1,N,79,4,2,9.30,2.50,0.5,0.00,0.0,1.0,16.55,2.5,0.0,0.75,None
3,2,2026-05-06 00:16:41,2026-05-06 00:22:50,1,1.75,1,N,140,263,1,9.30,1.00,0.5,2.86,0.0,1.0,17.16,2.5,0.0,0.00,None
4,1,2026-05-14 13:38:25,2026-05-14 13:57:54,2,1.50,1,N,262,229,1,17.00,3.25,0.5,3.50,0.0,1.0,25.25,2.5,0.0,0.75,None
5,2,2026-05-20 16:54:29,2026-05-20 17:06:54,<NA>,0.00,<NA>,NaN,68,246,0,23.47,0.00,0.5,0.00,0.0,1.0,27.47,NaN,NaN,0.00,None
6,2,2026-06-22 18:58:36,2026-06-22 19:39:04,1,17.31,4,N,138,265,1,86.30,7.50,0.5,19.46,0.0,1.0,116.76,0.0,2.0,0.00,None
7,1,2026-07-30 16:48:16,2026-07-30 16:48:40,1,0.20,5,N,216,132,1,105.00,0.00,0.0,10.00,0.0,1.0,116.00,0.0,0.0,0.00,None


In [9]:
correr('3_5b_muestra_green')

-- 3.5 Muestra de registros de taxis verdes (misma tecnica que 3.5a).
-- Fuente: data/raw/green/2026/*.parquet.
SELECT *
FROM read_parquet('/workspace/data/raw/green/2026/*.parquet', union_by_name = true)
WHERE hash(lpep_pickup_datetime, PULocationID, DOLocationID, total_amount) % 40000 = 0
ORDER BY lpep_pickup_datetime;



,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,request_source
0,2,2026-03-19 21:22:48,2026-03-19 21:26:50,N,1,33,52,1,0.78,5.8,1.0,0.5,0.83,0.0,NaN,1.0,9.13,1,1,0.0,0.0,None
1,2,2026-04-18 16:06:46,2026-04-18 16:16:48,N,1,166,152,1,1.04,10.7,0.0,0.5,1.83,0.0,NaN,1.0,14.03,1,1,0.0,0.0,None
2,6,2026-05-19 14:11:46,2026-05-19 14:40:15,NaN,<NA>,61,97,<NA>,3.16,0.0,0.0,0.5,0.00,0.0,NaN,0.3,20.16,<NA>,<NA>,NaN,0.0,None
3,2,2026-05-29 13:42:49,2026-05-29 13:59:39,N,1,130,130,1,2.31,17.0,0.0,0.5,2.00,0.0,NaN,1.0,20.50,1,1,0.0,0.0,None
4,2,2026-06-24 19:19:31,2026-06-24 19:22:11,N,1,74,42,1,1.37,7.9,2.5,0.5,2.38,0.0,NaN,1.0,14.28,1,1,0.0,0.0,None
5,2,2026-08-18 10:29:31,2026-08-18 10:37:48,N,1,74,42,1,1.56,10.7,0.0,0.5,0.00,0.0,NaN,1.0,12.20,2,1,0.0,0.0,None


## 3.6 Problemas de calidad de datos

Revisamos fechas, duraciones, distancias, montos, nulos, columnas que aparecen a mitad de año y posibles duplicados.

### Fechas fuera de rango

In [10]:
correr('3_6a_calidad_fechas')

-- 3.6 Calidad: fechas de recogida fuera de rango.
-- Objetivo: detectar viajes cuya fecha no corresponde al anio del archivo donde
-- vienen, o al mes de ese archivo.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) AS viajes,
    min(pickup) AS pickup_minimo,
    max(pickup) AS pickup_maximo,
    count(*) FILTER (WHERE pickup < make_date(anio_archivo, 1, 1)
                        OR pickup >= make_date(anio_archivo + 1, 1, 1)) AS fuera_del_anio_del_archivo,
    count(*) FILTER (WHERE strftime(pickup, '%Y-%m') <> mes_archivo) AS fuera_del_mes_del_archivo
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,viajes,pickup_minimo,pickup_maximo,fuera_del_anio_del_archivo,fuera_del_mes_del_archivo
0,green,337114,2008-12-31 17:35:31,2026-08-31 23:58:28,14,98
1,yellow,29703355,2001-01-01 09:23:58,2026-08-31 23:59:59,17,146


### Duraciones imposibles o absurdas

In [11]:
correr('3_6b_calidad_duracion')

-- 3.6 Calidad: duracion de los viajes.
-- Objetivo: contar viajes con duracion imposible o absurda (bajada antes de la
-- subida, duracion cero, mas de 6 horas) y ver la mediana y el percentil 99 de
-- la duracion en minutos de los viajes con duracion positiva.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) FILTER (WHERE dropoff < pickup) AS bajada_antes_de_subida,
    count(*) FILTER (WHERE dropoff = pickup) AS duracion_cero,
    count(*) FILTER (WHERE epoch(dropoff - pickup) > 6 * 3600) AS mas_de_6_horas,
    count(*) FILTER (WHERE epoch(dropoff - pickup) > 24 * 3600) AS mas_de_24_horas,
    round(quantile_cont(epoch(dropoff - pickup) / 60, 0.50)
          FILTER (WHERE dropoff > pickup), 1) AS mediana_min,
    round(quantile_cont(epoch(dropoff - pickup) / 60, 0.99)
          FILTER (WHERE dropoff > pickup), 1) AS p99_min
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,bajada_antes_de_subida,duracion_cero,mas_de_6_horas,mas_de_24_horas,mediana_min,p99_min
0,green,5,229,1104,4,13.1,82.3
1,yellow,10,371673,7315,263,14.1,71.4


### Distancias en cero o absurdas

In [12]:
correr('3_6c_calidad_distancia')

-- 3.6 Calidad: distancia de los viajes.
-- Objetivo: contar distancias en cero, negativas y absurdamente grandes.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) FILTER (WHERE trip_distance < 0) AS distancia_negativa,
    count(*) FILTER (WHERE trip_distance = 0) AS distancia_cero,
    count(*) FILTER (WHERE trip_distance > 100) AS mayor_a_100_millas,
    count(*) FILTER (WHERE trip_distance > 1000) AS mayor_a_1000_millas,
    max(trip_distance) AS distancia_maxima,
    round(quantile_cont(trip_distance, 0.50) FILTER (WHERE trip_distance > 0), 2) AS mediana_millas
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,distancia_negativa,distancia_cero,mayor_a_100_millas,mayor_a_1000_millas,distancia_maxima,mediana_millas
0,green,0,12212,72,66,179830.92,2.14
1,yellow,0,952231,1223,615,328522.20,1.92


### Montos negativos, en cero y extremos

In [13]:
correr('3_6d_calidad_montos')

-- 3.6 Calidad: montos de la tarifa y del total.
-- Objetivo: contar montos negativos, en cero y extremos, y ver cuanto pesan.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) FILTER (WHERE fare_amount < 0) AS tarifa_negativa,
    count(*) FILTER (WHERE fare_amount = 0) AS tarifa_cero,
    count(*) FILTER (WHERE total_amount < 0) AS total_negativo,
    count(*) FILTER (WHERE total_amount > 500) AS total_mayor_a_500,
    min(total_amount) AS total_minimo,
    max(total_amount) AS total_maximo,
    round(avg(total_amount), 2) AS total_promedio
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,tarifa_negativa,tarifa_cero,total_negativo,total_mayor_a_500,total_minimo,total_maximo,total_promedio
0,green,999,4880,1023,21,-501.5,1678.2,25.49
1,yellow,157364,22387,161835,791,-2560.2,7053.5,30.07


### Qué tipo de pago tienen los montos negativos

In [14]:
correr('3_6e_calidad_montos_negativos_por_pago')

-- 3.6 Calidad: a que tipo de pago pertenecen los montos negativos.
-- Objetivo: saber si los totales negativos son errores o devoluciones. Si se
-- concentran en payment_type 3 (sin cargo) y 4 (disputa), no son errores.
-- Codigos segun el diccionario de datos de la TLC: 0 Flex Fare, 1 tarjeta,
-- 2 efectivo, 3 sin cargo, 4 disputa, 5 desconocido, 6 viaje anulado.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    payment_type,
    count(*) AS viajes,
    count(*) FILTER (WHERE total_amount < 0) AS total_negativo,
    round(100.0 * count(*) FILTER (WHERE total_amount < 0) / count(*), 1) AS pct_negativo,
    round(avg(total_amount), 2) AS total_promedio
FROM viajes
GROUP BY tipo, payment_type
ORDER BY tipo, payment_type;



,tipo,payment_type,viajes,total_negativo,pct_negativo,total_promedio
0,green,1,219980,137,0.1,25.84
1,green,2,65921,43,0.1,21.25
2,green,3,1688,564,33.4,6.09
3,green,4,750,279,37.2,2.59
4,green,<NA>,48775,0,0.0,30.67
5,yellow,0,7716688,94,0.0,32.57
6,yellow,1,18941008,3511,0.0,30.18
7,yellow,2,2708031,39646,1.5,25.17
8,yellow,3,98138,17730,18.1,12.67
9,yellow,4,239488,100854,42.1,3.47


### Nulos y pasajeros fuera de rango

In [15]:
correr('3_6f_calidad_nulos')

-- 3.6 Calidad: valores nulos y pasajeros fuera de rango.
-- Objetivo: medir los nulos en las columnas que los tienen y comprobar si
-- ocurren juntos en las mismas filas (todos_nulos_juntos).
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) AS viajes,
    count(*) FILTER (WHERE passenger_count IS NULL) AS pasajeros_nulo,
    count(*) FILTER (WHERE RatecodeID IS NULL) AS ratecode_nulo,
    count(*) FILTER (WHERE store_and_fwd_flag IS NULL) AS flag_nulo,
    count(*) FILTER (WHERE payment_type IS NULL OR payment_type = 0) AS pago_nulo_o_cero,
    count(*) FILTER (WHERE passenger_count IS NULL AND RatecodeID IS NULL
                       AND store_and_fwd_flag IS NULL
                       AND coalesce(payment_type, 0) = 0) AS todos_nulos_juntos,
    count(*) FILTER (WHERE passenger_count = 0) AS pasajeros_cero,
    count(*) FILTER (WHERE passenger_count > 6) AS pasajeros_mayor_a_6
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,viajes,pasajeros_nulo,ratecode_nulo,flag_nulo,pago_nulo_o_cero,todos_nulos_juntos,pasajeros_cero,pasajeros_mayor_a_6
0,green,337114,48775,48775,48775,48775,48775,4527,99
1,yellow,29703355,7716688,7716688,7716688,7716688,7716688,91359,28


### Nulos de pasajeros por mes

In [16]:
correr('3_6g_calidad_nulos_por_mes')

-- 3.6 Calidad: peso de los nulos de pasajeros por mes y tipo.
-- Objetivo: ver si el problema de nulos es de un mes puntual o constante.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    mes_archivo,
    count(*) AS viajes,
    count(*) FILTER (WHERE passenger_count IS NULL) AS pasajeros_nulo,
    round(100.0 * count(*) FILTER (WHERE passenger_count IS NULL) / count(*), 1) AS pct_nulo
FROM viajes
GROUP BY tipo, mes_archivo
ORDER BY tipo, mes_archivo;



,tipo,mes_archivo,viajes,pasajeros_nulo,pct_nulo
0,green,2026-01,40272,5414,13.4
1,green,2026-02,37373,5387,14.4
2,green,2026-03,44208,6692,15.1
3,green,2026-04,44238,6290,14.2
4,green,2026-05,44921,5772,12.8
5,green,2026-06,44163,6472,14.7
6,green,2026-07,41252,6435,15.6
7,green,2026-08,40687,6313,15.5
8,yellow,2026-01,3724889,1088058,29.2
9,yellow,2026-02,3399866,1023317,30.1


### Columna `request_source`, que aparece desde junio

In [17]:
correr('3_6h_calidad_request_source')

-- 3.6 Calidad: columna request_source, que solo existe en algunos meses.
-- Objetivo: ver desde que mes aparece y que valores toma, porque en los meses
-- sin la columna el valor queda nulo al unir los archivos por nombre.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    mes_archivo,
    coalesce(request_source, '(nulo)') AS request_source,
    count(*) AS viajes
FROM viajes
WHERE mes_archivo >= '2026-05'
GROUP BY ALL
ORDER BY tipo, mes_archivo, viajes DESC;



,tipo,mes_archivo,request_source,viajes
0,green,2026-05,(nulo),44921
1,green,2026-06,(nulo),37692
2,green,2026-06,A,6471
3,green,2026-07,(nulo),34822
4,green,2026-07,A,6428
5,green,2026-07,CC,2
6,green,2026-08,(nulo),34377
7,green,2026-08,A,5130
8,green,2026-08,HV0005,1179
9,green,2026-08,CC,1


### Posibles duplicados (tarda unos 40 segundos)

In [18]:
correr('3_6i_calidad_duplicados')

-- 3.6 Calidad: posibles viajes duplicados.
-- Objetivo: estimar cuantas filas repiten exactamente el mismo viaje segun un
-- conjunto de columnas que lo identifica. Es una estimacion, porque dos viajes
-- distintos podrian coincidir en todas estas columnas. Tarda unos 40 segundos.
-- Fuente: vista viajes (00_vista_viajes.sql).
SELECT
    tipo,
    count(*) AS viajes,
    count(*) - count(DISTINCT (pickup, dropoff, PULocationID, DOLocationID,
                               total_amount, trip_distance, VendorID)) AS filas_repetidas
FROM viajes
GROUP BY tipo
ORDER BY tipo;



,tipo,viajes,filas_repetidas
0,green,337114,0
1,yellow,29703355,32790


## 3.7 Efecto de las reglas de limpieza

Con lo que encontramos en 3.6 probamos unas reglas simples de limpieza y medimos cuánto descartarían, antes de decidir si las aplicamos en el análisis exploratorio.

In [19]:
correr('3_7a_viajes_utilizables')

-- 3.7 Efecto de las reglas de limpieza candidatas.
-- Objetivo: medir cuantos viajes se descartarian al aplicar reglas simples de
-- calidad, para decidir si conviene filtrarlos en los analisis siguientes.
-- Reglas: pickup dentro del anio del archivo, duracion entre 1 minuto y 6 horas,
-- distancia mayor a 0 y menor o igual a 100 millas, total mayor a 0.
-- Fuente: vista viajes (00_vista_viajes.sql).
WITH evaluados AS (
    SELECT
        tipo,
        pickup >= make_date(anio_archivo, 1, 1) AND pickup < make_date(anio_archivo + 1, 1, 1) AS fecha_ok,
        epoch(dropoff - pickup) BETWEEN 60 AND 6 * 3600 AS duracion_ok,
        trip_distance > 0 AND trip_distance <= 100 AS distancia_ok,
        total_amount > 0 AS monto_ok
    FROM viajes
)
SELECT
    tipo,
    count(*) AS viajes,
    count(*) FILTER (WHERE fecha_ok AND duracion_ok AND distancia_ok AND monto_ok) AS utilizables,
    count(*) FILTER (WHERE NOT (fecha_ok AND duracion_ok AND distancia_ok AND monto_ok)) AS descartados,
 

,tipo,viajes,utilizables,descartados,pct_descartado,falla_fecha,falla_duracion,falla_distancia,falla_monto
0,green,337114,319257,17857,5.30,14,12380,12284,1566
1,yellow,29703355,28146188,1557167,5.24,17,688477,953454,167093
